In [ ]:
!pip install transformers torch accelerate

In [ ]:
!huggingface-cli login

/bin/bash: line 1: huggingface-cli: command not found


In [ ]:
!huggingface-cli whoami

/bin/bash: line 1: huggingface-cli: command not found


Loading Model & Tokenizer
Here, we are preparing our session by loading both the Llama model and its associated tokenizer.

The tokenizer will help in converting our text prompts into a format that the model can understand and process.

In [ ]:
from transformers import AutoTokenizer
import transformers
import torch
from huggingface_hub import login

# Log in to Hugging Face to access gated models
login()

model = "meta-llama/Llama-2-7b-chat-hf" # meta-llama/Llama-2-7b-hf

tokenizer = AutoTokenizer.from_pretrained(model, token=True)

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Creating the Llama Pipeline
We'll set up a pipeline for text generation.

This pipeline simplifies the process of feeding prompts to our model and receiving generated text as output.

Note: This cell takes 2-3 minutes to run

In [ ]:
from transformers import pipeline

llama_pipeline = pipeline(
    "text-generation",  # LLM task
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

#Getting Responses
With everything set up, let's see how Llama responds to some sample queries.

In [ ]:
def get_llama_response(prompt: str) -> None:
    """
    Generate a response from the Llama model.

    Parameters:
        prompt (str): The user's input/question for the model.

    Returns:
        None: Prints the model's response.
    """
    sequences = llama_pipeline(
        prompt,
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=256,
    )
    #print("Chatbot:", sequences[0]['generated_text'])
    return sequences[0]['generated_text']



prompt = 'I liked "Breaking Bad" and "Band of Brothers". Do you have any recommendations of other shows I might like?\n'
#get_llama_response(prompt)

In [ ]:

while True:
        user_input = input("User: ")

        if user_input.lower() in ["quit","bye","exit"]:
          print("Goodbye...")
          break

        prompt = user_input + "\n"
        print(f"ChatBot:",get_llama_response(prompt))

ChatBot: who was Maharana Pratap

Maharana Pratap (1540-1597) was a Rajput king of the Mewar region in western India, known for his bravery and resistance against the Mughal Empire during the reign of Emperor Akbar. He was the leader of the Mewar kingdom and is considered one of the greatest warriors of Indian history.

Early Life and Reign:
Maharana Pratap was born in 1540 in the city of Kumbhalgarh in the Mewar region of Rajasthan. He was the eldest son of Maharana Udai Singh II and Rani Jeevan Kanwar. After his father's death in 1548, Pratap became the king of Mewar at the age of eight.

Pratap's reign was marked by his resistance against the Mughal Empire, which had been expanding its territories in India since the 16th century. In 1576, Emperor Akbar, who had already conquered much of northern India, launched an attack on Mewar. Prat
ChatBot: who is uday singh

Uday Singh is a well-known Indian actor, producer, and television host who has been active in the entertainment industry 

In [ ]:
def chatbot():
    # Store conversation history
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["quit","bye","exit"]:
          print("Goodbye...")
          print(messages)
          break

        messages.append({"role": "user", "content": user_input})
        prompt = user_input + "\n"
        print(f"ChatBoT:",get_llama_response(prompt))
        messages.append({"role": "chatbot", "content": get_llama_response(prompt)})


if __name__ == "__main__":
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()

Start chatting with the bot (type 'quit' to stop)!
User: Who is Narendranath?
ChatBoT: Who is Narendranath?
Narendranath is a name that has been associated with several notable individuals throughout history. Here are some of the most well-known Narendranaths:

1. Swami Vivekananda: Narendranath Datta was born in Calcutta, India in 1863 and was later known as Swami Vivekananda, a prominent Hindu monk, philosopher, and spiritual leader. He is best known for his advocacy of Vedic philosophy and his contributions to the development of Hinduism in the modern era.
2. Narendranath Bhattacharya: Narendranath Bhattacharya was a Bengali writer, philosopher, and philologist who was born in 1881 and died in 1956. He was a prominent figure in the Bengali Renaissance and was known for his works on Bengali literature, philosophy, and culture.
3. Narendranath Dutta: Narendranath Dutta was an Indian independence activist and politician who was born in 1885 and died in 1974. He was a prominent leader i

In [ ]:
def chatbot():
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["quit", "bye", "exit"]:
            print("Goodbye...")
            break

        # Append user message to the conversation history
        messages.append({"role": "user", "content": user_input})

        # Get response from the Llama model by passing the full conversation history
        response_text = get_llama_response(messages)

        # Print the bot's response (which should be a string)
        print("Bot:", response_text)

        # Append the assistant's string response to the conversation history
        messages.append({"role": "assistant", "content": response_text})


def get_llama_response(current_messages: list) -> str:
    """
    Generate a response from the Llama model.

    Parameters:
        current_messages (list): The list of conversation messages for the model.

    Returns:
        str: The model's generated response.
    """
    sequences = llama_pipeline(
        current_messages, # Pass the entire message history for chat templating
        do_sample=True,
        top_k=10,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        max_length=256,
    )
    full_generated_text = sequences[0]['generated_text']

    # The pipeline returns the entire conversation including the input prompt.
    # We need to extract only the last assistant's reply.
    # For Llama-2 chat models, the assistant's response typically follows '[/INST]'.
    last_inst_idx = full_generated_text.rfind('[/INST]')

    if last_inst_idx != -1:
        # Extract the text after the last '[/INST]'
        assistant_response = full_generated_text[last_inst_idx + len('[/INST]'):].strip()

        # Remove any trailing '</s>' token if present
        if assistant_response.endswith(tokenizer.eos_token):
            assistant_response = assistant_response[:-len(tokenizer.eos_token)].strip()

        # If the model over-generates and starts a new turn, truncate it.
        new_turn_start_idx = assistant_response.find('<s>[INST]')
        if new_turn_start_idx != -1:
            assistant_response = assistant_response[:new_turn_start_idx].strip()

        return assistant_response
    else:
        # Fallback if '[/INST]' is not found (should not happen with Llama-2 chat models)
        return full_generated_text


if __name__ == "__main__":
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()

In [ ]:
#!pip install transformers torch acceleratefrom openai import OpenAI

# Initialize OpenAI client
#client = OpenAI(api_key="")


def chatbot():
    # Store conversation history
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["quit","bye","exit"]:
          print("Goodbye...")
          break

        # Add user message to memory
        messages.append({"role": "user", "content": user_input})

        response = client.responses.create(
            model="gpt-5.2",
            input=messages
        )

        # Extract assistant reply
        bot_reply = response.output_text
        print(f"Bot: {bot_reply}")

        # Add assistant reply back to memory
        messages.append({"role": "assistant", "content": bot_reply})


if __name__ == "__main__":
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()


NameError: name 'sequences' is not defined

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient()

def get_llama_response(messages):
    response = client.chat.completions.create(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=messages,
        max_tokens=256
    )
    return response.choices[0].message.content


def chatbot():
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["quit", "bye", "exit"]:
            print("Goodbye...")
            break

        messages.append({"role": "user", "content": user_input})

        response = get_llama_response(messages)

        print("Bot:", response)

        messages.append({"role": "assistant", "content": response})


if __name__ == "__main__":
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()


Start chatting with the bot (type 'quit' to stop)!
User: Narendra Modi and Swami Vivekanand's name
Bot: Narendra Modi and Swami Vivekananda are two prominent names from India, although not directly related, they both have made significant contributions to the country.

1. **Narendra Modi**: Narendra Damodardas Modi is an Indian politician who has been the Prime Minister of India since 2014. Born on September 17, 1950, in Vadnagar, Gujarat, Modi has played a vital role in India's economic growth and development. Prior to becoming Prime Minister, he served as the Chief Minister of Gujarat from 2001 to 2014. His government has implemented several initiatives aimed at promoting economic growth, infrastructure development, and social welfare programs. Modi is also known for his Hindu nationalist policies and strong stance on issues like terrorism and national security.

2. **Swami Vivekananda**: Swami Vivekananda (born Narendra Nath Datta on January 12, 1863) was an Indian philosopher, yogi

In [ ]:
def main():
    print("Start chatting with the bot (type 'quit' to stop)!")
    chatbot()